# 2048 Expectimax — Heuristic × Depth Experiments

Tests 4 heuristics across depths 1–4 (16 conditions, 5 games each = 80 games total).

**Heuristics:**
- `snake`   — positional snake-pattern weights only
- `empty`   — empty cell count only
- `smooth`  — smoothness (adjacent tile similarity) only
- `combined` — all three together (snake + empty + smooth)

## 1. Board logic

In [7]:
import random
import time

def empty_board():
    return [[0] * 4 for _ in range(4)]

def get_empty_cells(board):
    return [(r, c) for r in range(4) for c in range(4) if board[r][c] == 0]

def add_random_tile(board):
    empty = get_empty_cells(board)
    if not empty:
        return
    r, c = random.choice(empty)
    board[r][c] = 2 if random.random() < 0.9 else 4

def slide_row_left(row):
    tiles = [t for t in row if t != 0]
    merged, score = [], 0
    i = 0
    while i < len(tiles):
        if i + 1 < len(tiles) and tiles[i] == tiles[i + 1]:
            val = tiles[i] * 2
            merged.append(val)
            score += val
            i += 2
        else:
            merged.append(tiles[i])
            i += 1
    merged += [0] * (4 - len(merged))
    return merged, score

def rotate_cw(board):
    return [list(row) for row in zip(*board[::-1])]

def apply_move(board, direction):
    rot = {'left': 0, 'up': 1, 'right': 2, 'down': 3}[direction]
    b = [row[:] for row in board]
    for _ in range(rot):
        b = rotate_cw(b)
    new_b, total_score = [], 0
    for row in b:
        new_row, s = slide_row_left(row)
        new_b.append(new_row)
        total_score += s
    for _ in range((4 - rot) % 4):
        new_b = rotate_cw(new_b)
    if new_b == board:
        return None, 0
    return new_b, total_score

def is_game_over(board):
    return all(apply_move(board, d)[0] is None for d in ('left', 'right', 'up', 'down'))

print('Board logic ready.')

Board logic ready.


## 2. Heuristics

In [8]:
# Snake-pattern weight matrix
# High weights top-left, snaking across alternating rows
SNAKE_WEIGHTS = [
    [2**15, 2**14, 2**13, 2**12],
    [2**8,  2**9,  2**10, 2**11],
    [2**7,  2**6,  2**5,  2**4],
    [2**0,  2**1,  2**2,  2**3],
]

def heuristic_snake(board):
    """Positional snake-pattern score only."""
    return sum(
        board[r][c] * SNAKE_WEIGHTS[r][c]
        for r in range(4) for c in range(4)
    )

def heuristic_empty(board):
    """Empty cell count only (scaled to be comparable with other heuristics)."""
    return len(get_empty_cells(board)) * 10000

def heuristic_smooth(board):
    """Smoothness: penalise large differences between adjacent tiles."""
    penalty = 0
    for r in range(4):
        for c in range(4):
            v = board[r][c]
            if v == 0:
                continue
            if c + 1 < 4 and board[r][c + 1]:
                penalty -= abs(v - board[r][c + 1])
            if r + 1 < 4 and board[r + 1][c]:
                penalty -= abs(v - board[r + 1][c])
    return penalty

def heuristic_combined(board):
    """All three components together."""
    return heuristic_snake(board) + heuristic_empty(board) + heuristic_smooth(board)

# Registry — maps name -> function
HEURISTICS = {
    'snake':    heuristic_snake,
    'empty':    heuristic_empty,
    'smooth':   heuristic_smooth,
    'combined': heuristic_combined,
}

print('Heuristics ready:', list(HEURISTICS.keys()))

Heuristics ready: ['snake', 'empty', 'smooth', 'combined']


## 3. Expectimax + get_children

In [9]:
def get_children(state):
    """
    Returns list of (child_state, probability).
    state keys: 'board', 'is_max', 'last_score'
    """
    board = state['board']

    if state['is_max']:
        children = []
        for direction in ('left', 'right', 'up', 'down'):
            new_board, score = apply_move(board, direction)
            if new_board is not None:
                children.append((
                    {'board': new_board, 'is_max': False, 'last_score': score},
                    1.0
                ))
        return children

    else:  # chance node
        empty = get_empty_cells(board)
        if not empty:
            return []
        p_per_cell = 1.0 / len(empty)
        children = []
        for (r, c) in empty:
            for tile, tile_prob in [(2, 0.9), (4, 0.1)]:
                new_board = [row[:] for row in board]
                new_board[r][c] = tile
                children.append((
                    {'board': new_board, 'is_max': True, 'last_score': 0},
                    p_per_cell * tile_prob
                ))
        return children


def expectimax(state, depth, is_maximizer, eval_fn):
    """Returns (score, best_child_state)."""
    children = get_children(state)
    if depth == 0 or not children:
        return eval_fn(state['board']), None
    if is_maximizer:
        best_score, best_child = float('-inf'), None
        for child, _ in children:
            score, _ = expectimax(child, depth - 1, False, eval_fn)
            if score > best_score:
                best_score, best_child = score, child
        return best_score, best_child
    else:  # chance
        expected = 0.0
        for child, prob in children:
            score, _ = expectimax(child, depth - 1, True, eval_fn)
            expected += prob * score
        return expected, None


def best_move(board, depth, eval_fn):
    state = {'board': board, 'is_max': True, 'last_score': 0}
    _, child = expectimax(state, depth, True, eval_fn)
    return child


print('Expectimax ready.')

Expectimax ready.


## 4. Run experiments  *(4 heuristics × 4 depths × 5 games = 80 games)*

⚠️ Depth 4 games are slow (~30–120s each). Total runtime: roughly 30–90 minutes depending on your machine.

Progress is printed after every game so you can monitor.

In [ ]:
def play_game(depth, eval_fn, seed=None):
    """Play one full game. Returns dict with score, max_tile, moves."""
    if seed is not None:
        random.seed(seed)
    board = empty_board()
    add_random_tile(board)
    add_random_tile(board)
    score, moves = 0, 0
    while not is_game_over(board):
        result = best_move(board, depth, eval_fn)
        if result is None:
            break
        score += result['last_score']
        board = result['board']
        add_random_tile(board)
        moves += 1
    max_tile = max(board[r][c] for r in range(4) for c in range(4))
    return {'score': score, 'max_tile': max_tile, 'moves': moves}


N_GAMES = 25
DEPTHS  = [1, 2, 3]
SEEDS   = [i * 37 + 17 for i in range(N_GAMES)]  # 25 reproducible seeds

results = {}
total_conditions = len(HEURISTICS) * len(DEPTHS)
done = 0

for hname, hfn in HEURISTICS.items():
    for depth in DEPTHS:
        key = (hname, depth)
        results[key] = []
        t0 = time.time()
        for i, seed in enumerate(SEEDS):
            game = play_game(depth, hfn, seed=seed)
            results[key].append(game)
            print(f'  [{hname:>8} | depth={depth}] game {i+1}/{N_GAMES}  '
                  f'score={game["score"]:>7,}  max_tile={game["max_tile"]:>5}')
        elapsed = time.time() - t0
        done += 1
        print(f'  --> condition {done}/{total_conditions} done in {elapsed:.1f}s\n')

print('All experiments complete!')

  [   snake | depth=1] game 1/25  score=    756  max_tile=   64
  [   snake | depth=1] game 2/25  score=  1,856  max_tile=  128
  [   snake | depth=1] game 3/25  score=  3,688  max_tile=  256
  [   snake | depth=1] game 4/25  score=  2,080  max_tile=  128
  [   snake | depth=1] game 5/25  score=  4,176  max_tile=  256
  [   snake | depth=1] game 6/25  score=  1,504  max_tile=  128
  [   snake | depth=1] game 7/25  score=  1,800  max_tile=  128
  [   snake | depth=1] game 8/25  score=  3,032  max_tile=  256
  [   snake | depth=1] game 9/25  score=  1,736  max_tile=  128
  [   snake | depth=1] game 10/25  score=  2,004  max_tile=  128
  [   snake | depth=1] game 11/25  score=  1,496  max_tile=  128
  [   snake | depth=1] game 12/25  score=  2,504  max_tile=  256
  [   snake | depth=1] game 13/25  score=  3,852  max_tile=  256
  [   snake | depth=1] game 14/25  score=  5,236  max_tile=  512
  [   snake | depth=1] game 15/25  score=  2,432  max_tile=  128
  [   snake | depth=1] game 16/25 

## 5. Summary table

In [ ]:
from statistics import mean, stdev

def summarise(game_list):
    scores = [g['score'] for g in game_list]
    tiles  = [g['max_tile'] for g in game_list]
    return {
        'avg_score':  round(mean(scores)),
        'max_score':  max(scores),
        'min_score':  min(scores),
        'std_score':  round(stdev(scores)) if len(scores) > 1 else 0,
        'avg_tile':   round(mean(tiles)),
        'max_tile':   max(tiles),
        'rate_2048':  f"{sum(1 for t in tiles if t >= 2048)}/{len(tiles)}",
        'rate_1024':  f"{sum(1 for t in tiles if t >= 1024)}/{len(tiles)}",
    }

# Print a readable table
header = f"{'Heuristic':>10} | {'Depth':>5} | {'Avg':>7} | {'Max':>7} | {'Min':>7} | {'Std':>6} | {'MaxTile':>7} | {'>=2048':>6} | {'>=1024':>6}"
print(header)
print('-' * len(header))

summary = {}
for hname in HEURISTICS:
    for depth in DEPTHS:
        key = (hname, depth)
        s = summarise(results[key])
        summary[key] = s
        print(f"{hname:>10} | {depth:>5} | {s['avg_score']:>7,} | {s['max_score']:>7,} | "
              f"{s['min_score']:>7,} | {s['std_score']:>6,} | {s['max_tile']:>7} | "
              f"{s['rate_2048']:>6} | {s['rate_1024']:>6}")

 Heuristic | Depth |     Avg |     Max |     Min |    Std | MaxTile | >=2048 | >=1024
-------------------------------------------------------------------------------------
     snake |     1 |   2,426 |   3,536 |     772 |  1,078 |     256 |    0/5 |    0/5
     snake |     2 |   3,130 |   5,224 |     772 |  1,670 |     512 |    0/5 |    0/5
     snake |     3 |  30,610 |  59,368 |   7,156 | 20,176 |    4096 |    3/5 |    4/5
     empty |     1 |   3,556 |   5,648 |     740 |  1,915 |     512 |    0/5 |    0/5
     empty |     2 |   3,556 |   5,648 |     740 |  1,915 |     512 |    0/5 |    0/5
     empty |     3 |  11,199 |  16,196 |   5,452 |  5,408 |    1024 |    0/5 |    3/5
    smooth |     1 |   1,713 |   2,408 |   1,000 |    637 |     256 |    0/5 |    0/5
    smooth |     2 |   3,442 |   4,964 |   2,160 |  1,331 |     512 |    0/5 |    0/5
    smooth |     3 |   6,091 |  11,040 |   2,688 |  3,375 |     512 |    0/5 |    0/5
  combined |     1 |   2,565 |   3,108 |   1,896 |    

## 6. Copy results for the paper

Run this cell after experiments complete to get a clean paste-ready block of all results.

In [ ]:
print('RESULTS FOR PAPER')
print('=' * 60)
for hname in HEURISTICS:
    print(f'\nHeuristic: {hname}')
    for depth in DEPTHS:
        s = summary[(hname, depth)]
        print(f'  Depth {depth}: avg={s["avg_score"]:,}  max={s["max_score"]:,}  '
              f'min={s["min_score"]:,}  std={s["std_score"]:,}  '
              f'max_tile={s["max_tile"]}  2048_rate={s["rate_2048"]}  1024_rate={s["rate_1024"]}')

RESULTS FOR PAPER

Heuristic: snake
  Depth 1: avg=2,426  max=3,536  min=772  std=1,078  max_tile=256  2048_rate=0/5  1024_rate=0/5
  Depth 2: avg=3,130  max=5,224  min=772  std=1,670  max_tile=512  2048_rate=0/5  1024_rate=0/5
  Depth 3: avg=30,610  max=59,368  min=7,156  std=20,176  max_tile=4096  2048_rate=3/5  1024_rate=4/5

Heuristic: empty
  Depth 1: avg=3,556  max=5,648  min=740  std=1,915  max_tile=512  2048_rate=0/5  1024_rate=0/5
  Depth 2: avg=3,556  max=5,648  min=740  std=1,915  max_tile=512  2048_rate=0/5  1024_rate=0/5
  Depth 3: avg=11,199  max=16,196  min=5,452  std=5,408  max_tile=1024  2048_rate=0/5  1024_rate=3/5

Heuristic: smooth
  Depth 1: avg=1,713  max=2,408  min=1,000  std=637  max_tile=256  2048_rate=0/5  1024_rate=0/5
  Depth 2: avg=3,442  max=4,964  min=2,160  std=1,331  max_tile=512  2048_rate=0/5  1024_rate=0/5
  Depth 3: avg=6,091  max=11,040  min=2,688  std=3,375  max_tile=512  2048_rate=0/5  1024_rate=0/5

Heuristic: combined
  Depth 1: avg=2,565  max=